In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib
import os

# ============================================================
# DETECTION AUTO DU FICHIER CSV (recherche recursive)
# ============================================================
csv_files = []
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.csv'):
            csv_files.append(os.path.join(root, f))

print(f"Fichiers CSV trouvés sous /kaggle/input :")
for f in csv_files:
    print(f"  - {f}")

if not csv_files:
    raise FileNotFoundError(
        "Aucun CSV trouvé sous /kaggle/input.\n"
        f"Contenu racine : {os.listdir('/kaggle/input')}"
    )

# Prendre le premier CSV trouvé (ou change l'index si tu veux un fichier specifique)
csv_path = csv_files[0]
print(f"\nChargement de : {csv_path}")

# ============================================================
# CHARGEMENT CSV
# ============================================================
df = pd.read_csv(csv_path)

print(f"\nShape : {df.shape}")
print(f"Splits :\n{df['split'].value_counts()}")

# ============================================================
# FEATURES TABULAIRES EXACTES ECHONEXT
# ============================================================
feature_cols = [
    'sex',
    'ventricular_rate',
    'atrial_rate',
    'pr_interval',
    'qrs_duration',
    'qt_corrected',
    'age_at_ecg'
]

# Vérification avec diagnostic complet
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    print(f"\n Features manquantes : {missing}")
    print(f"Colonnes disponibles dans le CSV : {list(df.columns)}")
    raise ValueError("Certaines features n'existent pas dans le CSV !")

print("\nFeatures utilisées :")
print(feature_cols)

# ============================================================
# SPLITS
# ============================================================
train_mask = df['split'] == 'train'
val_mask   = df['split'] == 'val'
test_mask  = df['split'] == 'test'

# ============================================================
# EXTRACTION FEATURES
# ============================================================
X_train = df.loc[train_mask, feature_cols].values.astype(np.float32)
X_val   = df.loc[val_mask,   feature_cols].values.astype(np.float32)
X_test  = df.loc[test_mask,  feature_cols].values.astype(np.float32)

print("\nShapes avant standardisation :")
print(f"Train : {X_train.shape}")
print(f"Val   : {X_val.shape}")
print(f"Test  : {X_test.shape}")

# ============================================================
# VÉRIFICATION NaN
# ============================================================
print("\nNaN par feature :")
print(df[feature_cols].isnull().sum())

# ============================================================
# STANDARDISATION
# fit UNIQUEMENT sur train
# ============================================================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val   = scaler.transform(X_val).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

# ============================================================
# SAUVEGARDE FORMAT OFFICIEL ECHONEXT
# ============================================================
np.save(
    '/kaggle/working/EchoNext_train_tabular_features.npy',
    X_train
)

np.save(
    '/kaggle/working/EchoNext_val_tabular_features.npy',
    X_val
)

np.save(
    '/kaggle/working/EchoNext_test_tabular_features.npy',
    X_test
)

# sauvegarde scaler
joblib.dump(
    scaler,
    '/kaggle/working/tabular_scaler_7features.pkl'
)

# ============================================================
# VÉRIFICATION FINALE
# ============================================================
print("\n================================================")
print("FICHIERS SAUVEGARDÉS")
print("================================================")

print("✓ EchoNext_train_tabular_features.npy")
print("✓ EchoNext_val_tabular_features.npy")
print("✓ EchoNext_test_tabular_features.npy")
print("✓ tabular_scaler_7features.pkl")

print("\n================================================")
print("VÉRIFICATION")
print("================================================")

print(f"NaN train : {np.isnan(X_train).sum()}")
print(f"NaN val   : {np.isnan(X_val).sum()}")
print(f"NaN test  : {np.isnan(X_test).sum()}")

print(f"\nMean train : {X_train.mean():.4f}")
print(f"Std train  : {X_train.std():.4f}")

# ============================================================
# RELOAD TEST
# ============================================================
check = np.load(
    '/kaggle/working/EchoNext_train_tabular_features.npy'
)

print(f"\nShape finale train : {check.shape}")
print("Doit être : (N, 7)")

Fichiers CSV trouvés sous /kaggle/input :
  - /kaggle/input/datasets/nadakamel3/echonext-metada-preprocessed/echonext_metadata_clean.csv

Chargement de : /kaggle/input/datasets/nadakamel3/echonext-metada-preprocessed/echonext_metadata_clean.csv

Shape : (100000, 39)
Splits :
split
train       72475
no_split    17457
test         5442
val          4626
Name: count, dtype: int64

Features utilisées :
['sex', 'ventricular_rate', 'atrial_rate', 'pr_interval', 'qrs_duration', 'qt_corrected', 'age_at_ecg']

Shapes avant standardisation :
Train : (72475, 7)
Val   : (4626, 7)
Test  : (5442, 7)

NaN par feature :
sex                 0
ventricular_rate    0
atrial_rate         0
pr_interval         0
qrs_duration        0
qt_corrected        1
age_at_ecg          0
dtype: int64

FICHIERS SAUVEGARDÉS
✓ EchoNext_train_tabular_features.npy
✓ EchoNext_val_tabular_features.npy
✓ EchoNext_test_tabular_features.npy
✓ tabular_scaler_7features.pkl

VÉRIFICATION
NaN train : 1
NaN val   : 0
NaN test  : 0

